# KADİM — OCR + Analiz + RAG + Memur + Yazı (vLLM Multi-LoRA)

A100 80GB. **VL** ayrı vLLM. **Analiz + yazı** tek Instruct vLLM, iki LoRA.

Drive: `lora_adapter_qwen1`, `lora_adapter_qwen2`, `finetune_data_qwen1` (train.jsonl + rag + `belediye_konu.json`), VL klasörleri.

Dört ajan (Okuyucu → Analiz → Mevzuat → dur memur → Yazıcı). Birim tablodan, ajan değil. FastMCP 4 tool; dinleyen sunucu yok.

**Çalıştırma:** Run all → kırmızı RuntimeError → Restart → Run all. Gradio `share=True`.


## 1) Drive, yollar, rag.zip

Modeller Drive'da durur, lokal kopya yok. Bu hücre saniyeler sürmeli.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
BASE_DIR = DRIVE_ROOT / "finetune_data_qwen1"
MERGED_DIR = BASE_DIR / "merged_model_16bit"
RAG_ZIP = BASE_DIR / "rag.zip"
RAG_DIR_DRIVE = BASE_DIR / "rag"
if not RAG_DIR_DRIVE.is_dir() and (DRIVE_ROOT / "rag").is_dir():
    RAG_DIR_DRIVE = DRIVE_ROOT / "rag"
VL_BASE_DRIVE = DRIVE_ROOT / "qwen_vl_7b_model"
VL_LORA_DRIVE = DRIVE_ROOT / "qwen_vl_7b_lora_finetuned"
TRAIN_JSONL = BASE_DIR / "train.jsonl"
MAPS_JSON = BASE_DIR / "belediye_konu.json"
if not MAPS_JSON.is_file():
    for _alt in (
        DRIVE_ROOT / "belediye_konu.json",
        DRIVE_ROOT / "maps" / "belediye_konu.json",
    ):
        if _alt.is_file():
            MAPS_JSON = _alt
            break
assert MAPS_JSON.is_file(), (
    f"belediye_konu.json yok. Repo maps/belediye_konu.json dosyasini "
    f"{BASE_DIR} icine koy."
)

assert VL_BASE_DRIVE.is_dir() and any(VL_BASE_DRIVE.iterdir()), f"VL base yok/bos: {VL_BASE_DRIVE}"
assert (VL_LORA_DRIVE / "adapter_config.json").is_file(), f"VL LoRA yok: {VL_LORA_DRIVE}"
assert TRAIN_JSONL.is_file(), f"train.jsonl yok: {TRAIN_JSONL} (finetune_data_qwen1 icine koy)"

if not RAG_DIR_DRIVE.is_dir():
    assert RAG_ZIP.is_file(), (
        f"rag bulunamadi. {RAG_DIR_DRIVE} veya {RAG_ZIP} veya MyDrive/rag"
    )
    with zipfile.ZipFile(RAG_ZIP) as zf:
        zf.extractall(BASE_DIR)
    RAG_DIR_DRIVE = BASE_DIR / "rag"

if not (RAG_DIR_DRIVE / "search.py").is_file():
    found = None
    for cand in list(BASE_DIR.rglob("search.py")) + list(DRIVE_ROOT.rglob("search.py")):
        if cand.parent.name == "rag":
            found = cand.parent
            break
    assert found is not None, "rag/search.py bulunamadi"
    RAG_DIR_DRIVE = found

assert (RAG_DIR_DRIVE / "out" / "chunks.jsonl").is_file(), "rag/out/chunks.jsonl eksik"
assert (RAG_DIR_DRIVE / "out" / "vectors" / "bge-m3.npy").is_file(), "rag/out/vectors/bge-m3.npy eksik"

VL_BASE_DIR = str(VL_BASE_DRIVE)
VL_LORA_DIR = str(VL_LORA_DRIVE)
ANALYZER_DIR = str(MERGED_DIR) if MERGED_DIR.is_dir() else ""
RAG_DIR = str(RAG_DIR_DRIVE)

with open(os.path.join(VL_LORA_DIR, "adapter_config.json"), encoding="utf-8") as f:
    _adapter = json.load(f)
MAX_LORA_RANK = int(_adapter.get("r") or _adapter.get("lora_r") or 64)
print("VL LoRA rank:", MAX_LORA_RANK)
print("VL base:", VL_BASE_DIR)
print("VL lora:", VL_LORA_DIR)
print("RAG:", RAG_DIR)


def _has_adapter(d: Path) -> bool:
    if not (d / "adapter_config.json").is_file():
        return False
    return (d / "adapter_model.safetensors").is_file() or (d / "adapter_model.bin").is_file()

ANALYZER_LORA_DIR = None
for _d in [
    DRIVE_ROOT / "lora_adapter_qwen1",
    BASE_DIR / "lora_adapter_qwen1",
    BASE_DIR / "lora_adapter",
]:
    if _has_adapter(_d):
        ANALYZER_LORA_DIR = str(_d)
        break
assert ANALYZER_LORA_DIR, "Analiz LoRA yok: MyDrive/lora_adapter_qwen1 (adapter_config.json + safetensors)"

WRITER_LORA_DIR = None
for _d in [
    DRIVE_ROOT / "lora_adapter_qwen2",
    BASE_DIR / "lora_adapter_qwen2",
    DRIVE_ROOT / "lora_adapter",
]:
    if _has_adapter(_d) and str(_d.resolve()) != str(Path(ANALYZER_LORA_DIR).resolve()):
        WRITER_LORA_DIR = str(_d)
        break
assert WRITER_LORA_DIR, "Yazi LoRA yok: MyDrive/lora_adapter_qwen2 (adapter_config.json + safetensors)"

_base_cands = [
    DRIVE_ROOT / "Qwen2.5-7B-Instruct",
    BASE_DIR / "Qwen2.5-7B-Instruct",
    DRIVE_ROOT / "qwen_7b_instruct",
]
TEXT_BASE = "Qwen/Qwen2.5-7B-Instruct"
for _d in _base_cands:
    if _d.is_dir() and (_d / "config.json").is_file():
        TEXT_BASE = str(_d)
        break
WRITER_BASE = TEXT_BASE

with open(os.path.join(ANALYZER_LORA_DIR, "adapter_config.json"), encoding="utf-8") as f:
    _acfg = json.load(f)
with open(os.path.join(WRITER_LORA_DIR, "adapter_config.json"), encoding="utf-8") as f:
    _wcfg = json.load(f)
ANALYZER_LORA_RANK = int(_acfg.get("r") or 16)
WRITER_LORA_RANK = int(_wcfg.get("r") or 32)
TEXT_MAX_LORA_RANK = max(ANALYZER_LORA_RANK, WRITER_LORA_RANK, 16)
print("Analiz LoRA:", ANALYZER_LORA_DIR, "r=", ANALYZER_LORA_RANK, "base=", _acfg.get("base_model_name_or_path"))
print("Yazi LoRA:", WRITER_LORA_DIR, "r=", WRITER_LORA_RANK, "base=", _wcfg.get("base_model_name_or_path"))
print("Instruct taban (vLLM):", TEXT_BASE)
print("TEXT_MAX_LORA_RANK:", TEXT_MAX_LORA_RANK)


## 2) CONFIG


In [ ]:
import os, sys, json
from pathlib import Path

class _FilenoFix:
    def __init__(self, stream, fd):
        self._stream = stream
        self._fd = fd
    def __getattr__(self, name):
        return getattr(self._stream, name)
    def fileno(self):
        return self._fd

try:
    sys.stdout.fileno()
except Exception:
    sys.stdout = _FilenoFix(sys.stdout, 1)
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = _FilenoFix(sys.stderr, 2)

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Sirali yukleme: tek model icin yuksek util. Vision token siskinligi max_pixels + downscale ile kesilir.
GPU_MEM_UTIL = 0.88
MAX_MODEL_LEN_VL = 4096
MAX_MODEL_LEN_TEXT = 4096
MAX_NEW_TOKENS_OCR = 1280
MAX_NEW_TOKENS_ANALYSIS = 900
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_IMAGE_SIDE = 1280
RAG_TOP_K = 5
WORKER_TIMEOUT_SEC = 20 * 60
MAX_NEW_TOKENS_DRAFT = 640
VL_VRAM_HEADROOM_GB = 4.0
VL_UTIL_CAP = 0.42
TEXT_GPU_UTIL = 0.38  # Instruct + 2 LoRA; VL ile toplam ~0.80

OCR_PROMPT = (
    "Bu resmi belgenin TAMAMINI oku. Ustbaslik, antet, kurum adi, tarih, sayi, konu, "
    "hitap, govde, maddeler, imza, ek listesi, dipnot — hepsini sirasiyla, satirlari koruyarak yaz. "
    "Ortadaki bir paragrafı ozetleme veya atlama. Yorum, ceviri, duzeltme ekleme. "
    "Okunamayan yerlere [okunamadi] koy."
)

_maps_cands = []
if "MAPS_JSON" in globals():
    _maps_cands.append(Path(MAPS_JSON))
_maps_cands.extend([
    Path("/content/drive/MyDrive/finetune_data_qwen1/belediye_konu.json"),
    Path("/content/drive/MyDrive/belediye_konu.json"),
    Path("/content/maps/belediye_konu.json"),
    Path("/content/belediye_konu.json"),
])
_maps_path = next((p for p in _maps_cands if p.is_file()), None)
assert _maps_path is not None, (
    "belediye_konu.json yok. Repo maps/belediye_konu.json dosyasini "
    "Drive/finetune_data_qwen1 icine koy."
)
_maps = json.loads(_maps_path.read_text(encoding="utf-8"))
TOPIC_TO_UNIT = dict(_maps["topic_to_unit"])
TOPIC_TO_RAG = {
    k: " ".join(w for w in str(v).split() if not w.isdigit())
    for k, v in dict(_maps["topic_to_rag"]).items()
}
print("Harita:", _maps_path, "birim", len(_maps.get("units") or {}), "konu", len(TOPIC_TO_UNIT))

PROCESS_STATUS_CHOICES = [
    "INCELEMEDE",
    "TAMAMLANDI",
    "EKSIK_BILGI_BEKLENIYOR",
    "REDDEDILDI",
    "YONLENDIRILDI",
]

DRAFT_SYSTEM_PROMPT = r'''Sen belediye MUDURLUGUsun; vatandasa GIDECEK resmi cevabi yazarsin.
Gelen dilekceyi / OCR metnini TEKRAR YAZMA. Vatandas imzasi, TCKN, ev adresi, telefon YASAK.
Hitap vatandasa (Sayin + soyad yeterli). Antet belediye (target_unit). Kapanis mudurluk. "BELEDIYE BASKANLIGINA" yasak.
key_information PERSON / ad soyad BASVURAN vatandastir; yazi YAZARI degildir. O isimle imza, "ben", dilekce agzi YASAK. Sen mudurluksun, vatandasa CEVAP yazarsin.
Ciktiya performed_actions / process_status KOYMA; onlar GIRDI.

Sana analiz JSON, memur alanlari ve SEÇİLMİŞ mevzuat (selected_legislation) verilir.
SADECE gecerli JSON. Baska metin YOK.

CIKTI (sadece bu 4 alan; process_status / performed_actions / result_information / target_unit GIRDI'dir, sen uydurma veya ezme):
{
  "response_type": "BILGI_YAZISI",
  "target_unit": "girdi.target_unit aynen kopyala",
  "process_information": "Ic sistem takip notu (YALNIZCA TEK CUMLE, hitapsiz)",
  "draft": "T.C.\\n...tam resmi yazi..."
}

response_type SADECE:
BILGI_YAZISI, BILDIRIM_YAZISI, EKSIK_BILGI_BELGE_TAMAMLAMA_YAZISI, RET_YAZISI, YONLENDIRME_YAZISI, TESPIT_TUTANAGI, ONAY_YAZISI

process_status GIRDI'dir, memur seçer. Sen çıktıya process_status yazma.
Eger girdi process_status=EKSIK_BILGI_BEKLENIYOR ise response_type MUTLAKA EKSIK_BILGI_BELGE_TAMAMLAMA_YAZISI.
Bunun disinda EKSIK_BILGI_BELGE_TAMAMLAMA_YAZISI kullanma.

KURAL 1 - selected_legislation DOLUYSA atif GOVDEDE herhangi bir cumlede: kanun/yonetmelik ADI + madde NO. Kapanis/imza satirina veya yazinin en sonuna yigmak YASAK. "ilgili mevzuat hukumleri" tek basina YETMEZ.
Listede OLMAYAN kanun/yonetmelik/madde no YAZMA. Ezber atif YASAK; sadece JSON'daki kayitlari kopyala.
BOSSa veya RAG bos ise kanun adi / madde no UYDURMA; sadece genel "ilgili mevzuat hukumleri" de.
KURAL 2 - arz/rica: VATANDAS/OZEL_KURULUS -> rica (arz yasak). Ust makam -> arz. KAMU_KURUMU -> arz ve rica.
KURAL 3 - process_information TAM 1 CUMLE, hitapsiz, nokta ile biter.
KURAL 4 - requested_action vatandasin TALEBI, yapilmis is degil. performed_actions / result_information / process_status disinda asilama-kisirlastirma-toplama TAMAMLANDI diye UYDURMA. INCELEMEDE ise sadece kayit ve inceleme yaz.
draft KISA: antet + hitap + 3-4 kisa paragraf + kapanis. 80-120 kelime. Uzun gerekce, tekrar, mevzuat ozeti YASAK.
Ek / Ekler / ek listesi / "Basvuru Dilekcesi (1 sayfa)" YASAK. Vatandasa giden yazi EK BOLUMU icermez.
target_unit girdide gelir; sen birim SECMEZSIN, aynen kopyala.
Ajan kilidi (varsa) system mesajinin sonunda gelir; ona uy.
'''

print("Config hazir. GPU_MEM_UTIL=", GPU_MEM_UTIL, "MAX_LORA_RANK=", MAX_LORA_RANK)
print("Yazi LoRA dir:", WRITER_LORA_DIR)
print("Topic->birim kayit:", len(TOPIC_TO_UNIT), "RAG anahtar:", len(TOPIC_TO_RAG))


## 3) Kurulum

Colab'ın hazır `torchvision` / `torchaudio` / `torchcodec` paketleri vLLM sonrası CUDA tag'i ile çakışır.

**İlk Run all burada durur.** `RuntimeError` görünce **Restart session**, sonra tekrar **Run all**.


In [ ]:
import subprocess, sys

def _deps_ok() -> bool:
    try:
        import vllm, torchvision, sentence_transformers, peft, rank_bm25, einops, accelerate, gradio, PIL  # noqa: F401
        return True
    except Exception:
        return False

if _deps_ok():
    import torch, vllm
    print("Bagimliliklar mevcut, kurulum atlandi.")
    print("torch", torch.__version__, "cuda", torch.version.cuda, "vllm", getattr(vllm, "__version__", "?"))
else:
    print("=== paket kaldirma ===", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "torchcodec"])
    print("=== vLLM kurulumu ===", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "vllm"])
    import torch
    cuda_ver = (torch.version.cuda or "12.8").replace(".", "")
    cuda_tag = "cu" + cuda_ver[:3]
    print("=== torchvision", cuda_tag, "===", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-deps",
        "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}",
        "torchvision",
    ])
    print("=== RAG + Gradio bagimliliklari ===", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "sentence-transformers>=3.0", "peft>=0.14,<0.19", "accelerate>=1.0,<2",
        "einops", "rank_bm25", "pydantic", "tqdm", "gradio", "pillow", "qdrant-client",
        "fastmcp",
    ])
    raise RuntimeError(
        "KURULUM BITTI. Simdi: Runtime > Restart session, sonra tekrar Runtime > Run all. "
        "Bu hucre ikinci seferde atlanacak. Restart YAPMADAN devam etme."
    )


## 4) Worker script + pipeline config (vLLM ayrı süreçte)


In [ ]:
from pathlib import Path
WORKER_SRC = "#!/usr/bin/env python3\n\"\"\"Tek seferlik vLLM isi: OCR veya JSON analiz. Surec bitince GPU tamamen bosalir.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport traceback\n\nos.environ.setdefault(\"VLLM_ENABLE_V1_MULTIPROCESSING\", \"0\")\nos.environ.setdefault(\"TOKENIZERS_PARALLELISM\", \"false\")\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\n\ndef log(msg: str) -> None:\n    print(msg, flush=True)\n\n\ndef load_cfg() -> dict:\n    path = os.environ.get(\"PIPELINE_CONFIG\", \"/content/pipeline_config.json\")\n    with open(path, encoding=\"utf-8\") as f:\n        return json.load(f)\n\n\ndef build_llm(model: str, cfg: dict, multimodal: bool, enable_lora: bool):\n    from vllm import LLM\n\n    kwargs = dict(\n        model=model,\n        dtype=\"bfloat16\",\n        trust_remote_code=True,\n        max_num_seqs=1,\n        enforce_eager=True,\n        disable_log_stats=True,\n        gpu_memory_utilization=float(cfg[\"gpu_memory_utilization\"]),\n        max_model_len=int(cfg[\"max_model_len_vl\"] if multimodal else cfg[\"max_model_len_text\"]),\n    )\n    if multimodal:\n        kwargs[\"limit_mm_per_prompt\"] = {\"image\": 1}\n        kwargs[\"mm_processor_kwargs\"] = {\n            \"min_pixels\": int(cfg[\"min_pixels\"]),\n            \"max_pixels\": int(cfg[\"max_pixels\"]),\n        }\n        kwargs[\"mm_processor_cache_gb\"] = 0\n        if enable_lora:\n            kwargs[\"enable_lora\"] = True\n            kwargs[\"max_lora_rank\"] = int(cfg[\"max_lora_rank\"])\n            kwargs[\"max_loras\"] = 1\n\n    for attempt in range(3):\n        try:\n            return LLM(**kwargs)\n        except TypeError as e:\n            log(f\"LLM TypeError, kwargs sadele\u015ftiriliyor: {e}\")\n            msg = str(e)\n            if \"mm_processor_cache_gb\" in msg or \"mm_processor_cache_gb\" in kwargs:\n                kwargs.pop(\"mm_processor_cache_gb\", None)\n            if \"mm_processor_kwargs\" in msg or \"max_pixels\" in msg:\n                kwargs.pop(\"mm_processor_kwargs\", None)\n            if \"enforce_eager\" in msg:\n                kwargs.pop(\"enforce_eager\", None)\n            if \"disable_log_stats\" in msg:\n                kwargs.pop(\"disable_log_stats\", None)\n            if \"max_loras\" in msg:\n                kwargs.pop(\"max_loras\", None)\n        except Exception:\n            if multimodal and \"mm_processor_kwargs\" in kwargs:\n                log(\"mm_processor_kwargs ile y\u00fckleme ba\u015far\u0131s\u0131z, kald\u0131r\u0131l\u0131p tekrar denenecek.\")\n                traceback.print_exc()\n                kwargs.pop(\"mm_processor_kwargs\", None)\n                kwargs.pop(\"mm_processor_cache_gb\", None)\n                continue\n            raise\n    return LLM(**kwargs)\n\n\ndef generate_vl(llm, prompt_text, image, sampling, lora_request, ocr_prompt: str):\n    payloads = [\n        {\"prompt\": prompt_text, \"multi_modal_data\": {\"image\": image}},\n        {\"prompt\": prompt_text, \"multi_modal_data\": {\"image\": [image]}},\n    ]\n    last_err = None\n    for payload in payloads:\n        try:\n            gen_kwargs = {\"sampling_params\": sampling}\n            if lora_request is not None:\n                gen_kwargs[\"lora_request\"] = lora_request\n            outs = llm.generate([payload], **gen_kwargs)\n            return outs[0].outputs[0].text\n        except Exception as e:\n            last_err = e\n            log(f\"generate payload denemesi ba\u015far\u0131s\u0131z: {type(e).__name__}: {e}\")\n    try:\n        messages = [{\n            \"role\": \"user\",\n            \"content\": [\n                {\"type\": \"image_pil\", \"image_pil\": image},\n                {\"type\": \"text\", \"text\": ocr_prompt},\n            ],\n        }]\n        chat_kwargs = {\"sampling_params\": sampling}\n        if lora_request is not None:\n            chat_kwargs[\"lora_request\"] = lora_request\n        outs = llm.chat(messages, **chat_kwargs)\n        return outs[0].outputs[0].text\n    except Exception as e:\n        last_err = e\n        log(f\"chat() fallback ba\u015far\u0131s\u0131z: {type(e).__name__}: {e}\")\n    raise last_err\n\n\ndef cmd_ocr(args, cfg):\n    from PIL import Image\n    from transformers import AutoProcessor\n    from vllm import SamplingParams\n\n    image = Image.open(args.image).convert(\"RGB\")\n    processor = AutoProcessor.from_pretrained(cfg[\"vl_base_dir\"], trust_remote_code=True)\n    messages = [{\n        \"role\": \"user\",\n        \"content\": [\n            {\"type\": \"image\"},\n            {\"type\": \"text\", \"text\": cfg[\"ocr_prompt\"]},\n        ],\n    }]\n    prompt_text = processor.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True\n    )\n\n    lora_dir = cfg[\"vl_lora_dir\"]\n    use_lora = bool(lora_dir) and os.path.isfile(os.path.join(lora_dir, \"adapter_config.json\"))\n    log(f\"VL y\u00fckleniyor (lora={use_lora}) -> {cfg['vl_base_dir']}\")\n    llm = None\n    lora_request = None\n    try:\n        llm = build_llm(cfg[\"vl_base_dir\"], cfg, multimodal=True, enable_lora=use_lora)\n        if use_lora:\n            from vllm.lora.request import LoRARequest\n            lora_request = LoRARequest(\"qwen_vl_lora_finetuned\", 1, lora_dir)\n        sampling = SamplingParams(temperature=0.0, max_tokens=int(cfg[\"max_new_tokens_ocr\"]))\n        log(\"OCR \u00fcretimi ba\u015fl\u0131yor...\")\n        try:\n            text = generate_vl(llm, prompt_text, image, sampling, lora_request, cfg[\"ocr_prompt\"])\n        except Exception:\n            if lora_request is not None:\n                log(\"LoRA ile OCR ba\u015far\u0131s\u0131z, base VL ile tekrar deneniyor.\")\n                traceback.print_exc()\n                text = generate_vl(llm, prompt_text, image, sampling, None, cfg[\"ocr_prompt\"])\n            else:\n                raise\n        text = (text or \"\").strip()\n        with open(args.out, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"ok\": True, \"text\": text}, f, ensure_ascii=False)\n        log(f\"OCR bitti, {len(text)} karakter yaz\u0131ld\u0131.\")\n    finally:\n        del llm\n        try:\n            import gc\n            import torch\n            gc.collect()\n            torch.cuda.empty_cache()\n        except Exception:\n            pass\n\n\ndef cmd_analyze(args, cfg):\n    from transformers import AutoTokenizer\n    from vllm import SamplingParams\n\n    with open(args.text_file, encoding=\"utf-8\") as f:\n        document_text = f.read()\n    with open(cfg[\"system_prompt_path\"], encoding=\"utf-8\") as f:\n        system_prompt = f.read()\n\n    tokenizer = AutoTokenizer.from_pretrained(cfg[\"analyzer_dir\"], trust_remote_code=True)\n    messages = [\n        {\"role\": \"system\", \"content\": system_prompt},\n        {\"role\": \"user\", \"content\": document_text},\n    ]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    log(f\"Analiz modeli y\u00fckleniyor -> {cfg['analyzer_dir']}\")\n    llm = None\n    try:\n        llm = build_llm(cfg[\"analyzer_dir\"], cfg, multimodal=False, enable_lora=False)\n        sampling = SamplingParams(temperature=0.0, max_tokens=int(cfg[\"max_new_tokens_analysis\"]))\n        log(\"JSON analiz \u00fcretimi ba\u015fl\u0131yor...\")\n        out = llm.generate([prompt], sampling)[0].outputs[0].text\n        with open(args.out, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"ok\": True, \"text\": out}, f, ensure_ascii=False)\n        log(\"Analiz bitti.\")\n    finally:\n        del llm\n        try:\n            import gc\n            import torch\n            gc.collect()\n            torch.cuda.empty_cache()\n        except Exception:\n            pass\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"task\", choices=[\"ocr\", \"analyze\"])\n    parser.add_argument(\"--image\")\n    parser.add_argument(\"--text_file\")\n    parser.add_argument(\"--out\", required=True)\n    args = parser.parse_args()\n    cfg = load_cfg()\n    try:\n        if args.task == \"ocr\":\n            if not args.image:\n                raise SystemExit(\"--image gerekli\")\n            cmd_ocr(args, cfg)\n        else:\n            if not args.text_file:\n                raise SystemExit(\"--text_file gerekli\")\n            cmd_analyze(args, cfg)\n    except Exception:\n        traceback.print_exc()\n        with open(args.out, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"ok\": False, \"error\": traceback.format_exc()}, f, ensure_ascii=False)\n        sys.exit(1)\n\n\nif __name__ == \"__main__\":\n    main()\n"
WORKER_PATH = '/content/vllm_worker.py'
CONFIG_PATH = '/content/pipeline_config.json'
SYSTEM_PROMPT_PATH = '/content/system_prompt.txt'

with open(BASE_DIR / 'train.jsonl', encoding='utf-8') as f:
    SYSTEM_PROMPT_INFERENCE = json.loads(f.readline())['messages'][0]['content']
Path(SYSTEM_PROMPT_PATH).write_text(SYSTEM_PROMPT_INFERENCE, encoding='utf-8')
print('Sistem promptu:', SYSTEM_PROMPT_INFERENCE[:180].replace('\n', ' '), '...')

cfg = {
    'vl_base_dir': VL_BASE_DIR,
    'vl_lora_dir': VL_LORA_DIR,
    'analyzer_dir': ANALYZER_DIR,
    'system_prompt_path': SYSTEM_PROMPT_PATH,
    'gpu_memory_utilization': GPU_MEM_UTIL,
    'max_model_len_vl': MAX_MODEL_LEN_VL,
    'max_model_len_text': MAX_MODEL_LEN_TEXT,
    'max_new_tokens_ocr': MAX_NEW_TOKENS_OCR,
    'max_new_tokens_analysis': MAX_NEW_TOKENS_ANALYSIS,
    'min_pixels': MIN_PIXELS,
    'max_pixels': MAX_PIXELS,
    'max_lora_rank': MAX_LORA_RANK,
    'ocr_prompt': OCR_PROMPT,
}
Path(CONFIG_PATH).write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
Path(WORKER_PATH).write_text(WORKER_SRC, encoding='utf-8')
print('Yazildi:', WORKER_PATH, CONFIG_PATH)


## 5) RAG'ı CPU'da ısıt

Embedder + reranker GPU'da kalırsa sonraki vLLM süreci OOM alır. Bu yüzden bilinçli olarak CPU'da tutulur (~4GB RAM, A100 RAM'i yeter).


In [ ]:
import inspect
import sys

rag_path = Path(RAG_DIR)
rag_parent = str(rag_path.parent)
if rag_parent not in sys.path:
    sys.path.insert(0, rag_parent)
init_py = rag_path / "__init__.py"
if not init_py.exists():
    init_py.write_text("", encoding="utf-8")

from rag.search import Searcher

_init = inspect.signature(Searcher.__init__)
_kwargs = {}
for key, val in [("device", "cpu"), ("embed_device", "cpu"), ("rerank_device", "cpu")]:
    if key in _init.parameters:
        _kwargs[key] = val

print("Searcher kwargs:", _kwargs or "(yok, varsayilan)")
searcher = Searcher(**_kwargs)
if hasattr(searcher, "warmup"):
    searcher.warmup()
print("RAG dummy arama (ilk belgeyi hizlandirir)...", flush=True)
try:
    searcher.search(["belediye gürültü denetim yaptırım"], top_k=3, abstain=False)
    print("RAG dummy tamam.")
except Exception as _e:
    print("RAG dummy atlandi:", _e)

# Ne olursa olsun GPU'da kalmasin
for attr in dir(searcher):
    if attr.startswith("_"):
        continue
    obj = getattr(searcher, attr, None)
    if obj is None:
        continue
    if hasattr(obj, "to"):
        try:
            obj.to("cpu")
        except Exception:
            pass
    inner = getattr(obj, "model", None) or getattr(obj, "_modules", None)
    if inner is not None and hasattr(inner, "to"):
        try:
            inner.to("cpu")
        except Exception:
            pass

n_chunks = len(getattr(searcher, "chunks", []) or [])
print(f"RAG hazir (CPU). parca={n_chunks} retriever={getattr(searcher, 'retriever', '?')}")
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU bos: {free/1024**3:.2f} / {total/1024**3:.2f} GB")
except Exception as e:
    print("GPU bellek okunamadi:", e)


## 6) Modelleri bir kez yükle (VL + analiz + yazı) + pipeline

80GB: analiz ve yazı transformers, VL vLLM kalan bütçe. 5–10 dk sürebilir. Bitince jüri belgelerinde model yüklenmez.


In [ ]:
import gc
import re
import subprocess
import tempfile
import time
import traceback
from pathlib import Path

import torch
from PIL import Image


def try_parse_json(text: str):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    candidate = text[text.find("{"):] if "{" in text else text
    if candidate.count('"') % 2 == 1:
        candidate += '"'
    opens_curly = candidate.count("{") - candidate.count("}")
    opens_square = candidate.count("[") - candidate.count("]")
    candidate = candidate.rstrip().rstrip(",")
    candidate += "]" * max(0, opens_square) + "}" * max(0, opens_curly)
    try:
        return json.loads(candidate)
    except Exception:
        return None


def downscale_image(image: Image.Image, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    image = image.convert("RGB")
    w, h = image.size
    m = max(w, h)
    if m <= max_side:
        return image
    scale = max_side / m
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resample = getattr(getattr(Image, "Resampling", Image), "LANCZOS", Image.LANCZOS)
    return image.resize((nw, nh), resample)


def _gpu_report(tag: str = ""):
    if not torch.cuda.is_available():
        print("CUDA yok")
        return 0, 0
    free, total = torch.cuda.mem_get_info()
    print(f"GPU {tag}: bos={free/1024**3:.2f}GB  toplam={total/1024**3:.2f}GB", flush=True)
    return free, total


def _run_worker(task: str, extra_args: list, status_cb=None) -> str:
    out_path = Path(tempfile.mkstemp(prefix=f"{task}_", suffix=".json")[1])
    cmd = [sys.executable, WORKER_PATH, task, *extra_args, "--out", str(out_path)]
    env = os.environ.copy()
    env["PIPELINE_CONFIG"] = CONFIG_PATH
    env["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    env["TOKENIZERS_PARALLELISM"] = "false"
    if status_cb:
        status_cb(f"{task}: vLLM süreci başlıyor (model yükleme 1–2 dk)...")
    print("RUN:", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=WORKER_TIMEOUT_SEC)
    if proc.stdout:
        print(proc.stdout[-8000:], flush=True)
    if proc.stderr:
        print(proc.stderr[-8000:], flush=True)
    if not out_path.is_file():
        raise RuntimeError(f"{task} çıktı dosyası yok. returncode={proc.returncode}\n{proc.stderr[-4000:]}")
    payload = json.loads(out_path.read_text(encoding="utf-8"))
    try:
        out_path.unlink()
    except Exception:
        pass
    if not payload.get("ok"):
        raise RuntimeError(payload.get("error") or f"{task} başarısız, rc={proc.returncode}")
    return payload["text"]


def _build_vl_llm(util: float):
    from vllm import LLM
    kwargs = dict(
        model=VL_BASE_DIR,
        dtype="bfloat16",
        trust_remote_code=True,
        max_num_seqs=1,
        enforce_eager=False,
        disable_log_stats=True,
        gpu_memory_utilization=float(util),
        max_model_len=MAX_MODEL_LEN_VL,
        limit_mm_per_prompt={"image": 1},
        enable_lora=True,
        max_lora_rank=MAX_LORA_RANK,
        max_loras=1,
        mm_processor_kwargs={"min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
        mm_processor_cache_gb=0,
    )
    try:
        return LLM(**kwargs)
    except TypeError as e:
        print("VL LLM TypeError, sadeleştiriliyor:", e, flush=True)
        kwargs.pop("mm_processor_cache_gb", None)
        kwargs.pop("disable_log_stats", None)
        if "mm_processor_kwargs" in str(e) or "max_pixels" in str(e):
            kwargs.pop("mm_processor_kwargs", None)
        if "max_loras" in str(e):
            kwargs.pop("max_loras", None)
        return LLM(**kwargs)


def _build_text_llm(util: float):
    from vllm import LLM
    kwargs = dict(
        model=TEXT_BASE,
        dtype="bfloat16",
        trust_remote_code=True,
        max_num_seqs=1,
        enforce_eager=False,
        disable_log_stats=True,
        gpu_memory_utilization=float(util),
        max_model_len=MAX_MODEL_LEN_TEXT,
        enable_lora=True,
        max_lora_rank=int(TEXT_MAX_LORA_RANK),
        max_loras=2,
    )
    try:
        return LLM(**kwargs)
    except TypeError as e:
        print("text LLM TypeError:", e, flush=True)
        kwargs.pop("disable_log_stats", None)
        if "max_loras" in str(e):
            kwargs.pop("max_loras", None)
        return LLM(**kwargs)


def load_models_once():
    """Instruct vLLM + 2 LoRA, sonra VL vLLM. Merge/transformers 7B yok."""
    global MODELS_READY, USE_SEQUENTIAL
    global text_llm, text_tokenizer, analiz_lora_request, yazi_lora_request
    global vl_llm, vl_processor, vl_lora_request, vl_sampling
    global analyzer_model, writer_model
    analyzer_model = None
    writer_model = None

    if globals().get("MODELS_READY"):
        print("Modeller zaten yüklü — tekrar yüklenmeyecek.")
        return

    from transformers import AutoProcessor, AutoTokenizer
    from vllm import SamplingParams
    from vllm.lora.request import LoRARequest

    _gpu_report("baslangic")
    print("1/2 Instruct vLLM (multi-LoRA analiz+yazi)...", flush=True)
    last_err = None
    text_llm = None
    for util in [float(TEXT_GPU_UTIL), 0.34, 0.30]:
        print(f"  text util={util:.3f}", flush=True)
        try:
            text_llm = _build_text_llm(util)
            last_err = None
            break
        except Exception as e:
            last_err = e
            print("  text vLLM basarisiz:", type(e).__name__, e, flush=True)
            gc.collect()
            torch.cuda.empty_cache()
    if text_llm is None:
        MODELS_READY = False
        USE_SEQUENTIAL = True
        print("Instruct vLLM yuklenemedi. Son hata:", last_err)
        return

    text_tokenizer = AutoTokenizer.from_pretrained(TEXT_BASE, trust_remote_code=True)
    analiz_lora_request = LoRARequest("analiz_lora", 1, ANALYZER_LORA_DIR)
    yazi_lora_request = LoRARequest("yazi_lora", 2, WRITER_LORA_DIR)
    _gpu_report("instruct sonrasi")

    free, total = torch.cuda.mem_get_info()
    head = float(VL_VRAM_HEADROOM_GB) * 1024**3
    auto_util = (free - head) / total
    remain_cap = max(0.30, min(float(VL_UTIL_CAP), 0.90 - float(TEXT_GPU_UTIL)))
    auto_util = max(0.28, min(float(VL_UTIL_CAP), remain_cap, auto_util))
    print(f"2/2 VL vLLM, gpu_memory_utilization={auto_util:.3f}", flush=True)

    vl_llm = None
    for util in [auto_util, auto_util - 0.08, 0.38, 0.32]:
        util = float(max(0.28, util))
        print(f"  VL util={util:.3f}", flush=True)
        try:
            vl_llm = _build_vl_llm(util)
            last_err = None
            break
        except Exception as e:
            last_err = e
            print("  VL yukleme basarisiz:", type(e).__name__, e, flush=True)
            gc.collect()
            torch.cuda.empty_cache()

    if vl_llm is None:
        print("VL sigmadi. text vLLM duruyor, OCR worker'a duser.", flush=True)
        MODELS_READY = False
        USE_SEQUENTIAL = True
        if last_err:
            print("Son VL hatasi:", last_err)
        return

    vl_processor = AutoProcessor.from_pretrained(VL_BASE_DIR, trust_remote_code=True)
    vl_lora_request = LoRARequest("qwen_vl_lora_finetuned", 1, VL_LORA_DIR)
    vl_sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS_OCR)
    MODELS_READY = True
    USE_SEQUENTIAL = False
    _gpu_report("iki vLLM hazir")
    print("Instruct (2 LoRA) + VL resident. Sonraki belgeler model YUKLEMEZ.", flush=True)


def _text_vllm(messages: list, lora_request, max_tokens: int) -> str:
    if not globals().get("text_llm"):
        return "(instruct vLLM yuklu degil)"
    from vllm import SamplingParams
    prompt = text_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    sp = SamplingParams(temperature=0.0, max_tokens=int(max_tokens))
    outs = text_llm.generate([prompt], sampling_params=sp, lora_request=lora_request)
    return (outs[0].outputs[0].text or "").strip()


def _ocr_resident(image: Image.Image) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": OCR_PROMPT},
        ],
    }]
    prompt_text = vl_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    payloads = [
        {"prompt": prompt_text, "multi_modal_data": {"image": image}},
        {"prompt": prompt_text, "multi_modal_data": {"image": [image]}},
    ]
    last_err = None
    for payload in payloads:
        try:
            outs = vl_llm.generate([payload], sampling_params=vl_sampling, lora_request=vl_lora_request)
            return (outs[0].outputs[0].text or "").strip()
        except Exception as e:
            last_err = e
    try:
        messages2 = [{
            "role": "user",
            "content": [
                {"type": "image_pil", "image_pil": image},
                {"type": "text", "text": OCR_PROMPT},
            ],
        }]
        outs = vl_llm.chat(messages2, sampling_params=vl_sampling, lora_request=vl_lora_request)
        return (outs[0].outputs[0].text or "").strip()
    except Exception as e:
        last_err = e
    raise last_err


def _analyze_resident(document_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_INFERENCE},
        {"role": "user", "content": document_text},
    ]
    return _text_vllm(messages, analiz_lora_request, MAX_NEW_TOKENS_ANALYSIS)



def _provisions_plain(provisions: list) -> str:
    if not provisions:
        return "(mevzuat bulunamadi)"
    parts = []
    for i, p in enumerate(provisions[:6], 1):
        kanun = getattr(p, "kanun", "") or (p.get("kanun") if isinstance(p, dict) else "")
        madde = getattr(p, "madde", "") or (p.get("madde") if isinstance(p, dict) else "")
        baslik = getattr(p, "baslik", "") or (p.get("baslik") if isinstance(p, dict) else "")
        metin = getattr(p, "metin", "") or (p.get("metin") if isinstance(p, dict) else "")
        parts.append(f"{i}. {kanun} {madde} {baslik}\n{str(metin)[:500]}")
    return "\n\n".join(parts)


def lookup_unit(analysis: dict) -> str:
    topic = str((analysis or {}).get("primary_topic") or "").strip()
    return TOPIC_TO_UNIT.get(topic, "Yazı İşleri Müdürlüğü")


def _slim_selected(payload: dict) -> list:
    """Yaziciya madde metni gitmesin; icindeki baska kanun adlari ezberi tetikliyor."""
    out = []
    for x in (payload or {}).get("selected_legislation") or []:
        if not isinstance(x, dict):
            continue
        item = {
            "kanun": (x.get("kanun") or "").strip(),
            "madde": (x.get("madde") or "").strip(),
            "baslik": (x.get("baslik") or "").strip(),
        }
        if item["kanun"] or item["madde"]:
            out.append(item)
    return out


def _draft_resident(ocr_text: str, payload: dict, extra_system: str = "") -> str:
    """payload = Qwen1 alanlari + memur girdileri + selected_legislation. Model birim secmez."""
    if not globals().get("text_llm"):
        return "(instruct vLLM yuklu degil)"
    user_obj = dict(payload or {})
    user_obj.pop("ocr_excerpt", None)
    slim = _slim_selected(payload)
    user_obj["selected_legislation"] = slim
    user_obj["_rol"] = "belediye cevabi; dilekce kopyalama yasak"
    cites = []
    for x in slim:
        kanun, madde = x["kanun"], x["madde"]
        baslik = x.get("baslik") or ""
        if madde and "madde" not in madde.lower() and any(ch.isdigit() for ch in madde):
            madde = f"Madde {madde}"
        lab = f"{kanun} {madde}".strip()
        if baslik and baslik not in {"-", "—"} and baslik.lower() not in lab.lower():
            lab = f"{lab} ({baslik})"
        if lab:
            cites.append(lab)
    must = ""
    if cites:
        must = (
            "ZORUNLU ATIF — GOVDEDE herhangi bir cumlede yaz (en sona yigma): "
            + "; ".join(cites)
            + "\n"
        )
    else:
        must = "selected_legislation BOS. Kanun/madde UYDURMA.\n"
    user = (
        must
        + "GIRDI JSON (Qwen1 analizi + memur alanlari + selected_legislation).\n"
        + "process_status, performed_actions, result_information, target_unit GIRDI'dir; UYDURMA.\n"
        + "CIKTI SADECE JSON: response_type, target_unit (girdiyi kopyala), process_information, draft.\n\n"
        + json.dumps(user_obj, ensure_ascii=False, indent=2)
    )
    sys_p = DRAFT_SYSTEM_PROMPT.rstrip()
    if extra_system:
        sys_p = sys_p + "\n\n" + extra_system
    messages = [
        {"role": "system", "content": sys_p},
        {"role": "user", "content": user},
    ]
    return _text_vllm(messages, yazi_lora_request, MAX_NEW_TOKENS_DRAFT)


def ocr_image(image: Image.Image, status_cb=None) -> str:
    image = downscale_image(image)
    if globals().get("MODELS_READY"):
        return _ocr_resident(image)
    tmp = Path(tempfile.mkstemp(prefix="ocr_img_", suffix=".png")[1])
    image.save(tmp, format="PNG")
    try:
        return _run_worker("ocr", ["--image", str(tmp)], status_cb=status_cb)
    finally:
        try:
            tmp.unlink()
        except Exception:
            pass
        gc.collect()


def analyze_text(document_text: str, status_cb=None) -> dict:
    if globals().get("MODELS_READY"):
        raw = _analyze_resident(document_text)
    else:
        tmp = Path(tempfile.mkstemp(prefix="doc_", suffix=".txt")[1])
        tmp.write_text(document_text, encoding="utf-8")
        try:
            raw = _run_worker("analyze", ["--text_file", str(tmp)], status_cb=status_cb)
        finally:
            try:
                tmp.unlink()
            except Exception:
                pass
            gc.collect()
    parsed = try_parse_json(raw)
    if parsed is None:
        return {"_parse_error": True, "_raw": raw}
    return parsed


def _complaint_bit(analysis: dict) -> str:
    ki = analysis.get("key_information") or {}
    if isinstance(ki, dict):
        for key in ("konu", "sikayet", "olay", "talep", "ozet", "özet"):
            val = ki.get(key)
            if isinstance(val, str) and len(val.strip()) > 12:
                return val.strip()[:240]
        for val in ki.values():
            if isinstance(val, str) and len(val.strip()) > 20:
                return val.strip()[:240]
            if isinstance(val, list) and val:
                return str(val[0]).strip()[:240]
    sm = str(analysis.get("summary") or "").strip()
    if not sm:
        return ""
    return sm.split(".")[0].strip()[:240]


def _strip_rag_numbers(q: str) -> str:
    """Map JSON / action icindeki 5199, 4982 gibi kanun no sizintisini kes."""
    q = re.sub(r"\b\d{3,5}\b", " ", q or "")
    return re.sub(r"\s+", " ", q).strip()


def build_rag_queries(analysis: dict) -> list:
    """Sadece topic Turkce anahtar + requested_action. OCR/ozet/key_information yok. Kanun no yok."""
    seen, out = set(), []
    def add(q):
        q = _strip_rag_numbers(q)
        if len(q) < 8:
            return
        key = q.lower()
        if key in seen:
            return
        seen.add(key)
        out.append(q)

    topic = str(analysis.get("primary_topic") or "").strip()
    add(TOPIC_TO_RAG.get(topic) or "")
    action = str(analysis.get("requested_action") or "").strip()
    al = action.lower()
    addr_hits = sum(
        1
        for a in ("mahallesi", "caddesi", "sokak", "bulvar", " no:", "no.", "cad.", "mah.")
        if a in al
    )
    if addr_hits < 2:
        add(action)
    if out[:2]:
        return out[:2]
    fb = _strip_rag_numbers(TOPIC_TO_RAG.get(topic) or "belediye gorev yetki mevzuat")
    return [fb]


def is_boilerplate_article(madde: str, baslik: str) -> bool:
    m = (madde or "").lower()
    b = (baslik or "").lower().strip()
    if "geçici" in m or "gecici" in m:
        return True
    if b in {"amaç", "amac", "kapsam", "dayanak", "tanımlar", "tanimlar", "tanım", "tanim"}:
        return True
    if b.startswith("amaç") or b.startswith("amac") or b.startswith("tanım") or b.startswith("tanim"):
        return True
    return False


def filter_boilerplate_provisions(provisions: list) -> list:
    kept = []
    for p in provisions or []:
        if isinstance(p, dict):
            madde, baslik = p.get("madde") or "", p.get("baslik") or ""
        else:
            madde = getattr(p, "madde", "") or ""
            baslik = getattr(p, "baslik", "") or ""
        if is_boilerplate_article(madde, baslik):
            continue
        kept.append(p)
    return kept


def provisions_to_markdown(queries: list, provisions: list) -> str:
    if not provisions:
        neden = "bilinmiyor"
        last = getattr(searcher, "last_decision", None)
        if last is not None:
            neden = getattr(last, "neden", None) or getattr(last, "reason", None) or str(last)
        return f"İlgili mevzuat bulunamadı (neden: {neden})."
    lines = [f"**Kullanılan sorgular:** {'; '.join(queries)}", ""]
    for i, p in enumerate(provisions, 1):
        kanun = getattr(p, "kanun", "") or (p.get("kanun") if isinstance(p, dict) else "")
        madde = getattr(p, "madde", "") or (p.get("madde") if isinstance(p, dict) else "")
        baslik = getattr(p, "baslik", "") or (p.get("baslik") if isinstance(p, dict) else "")
        metin = getattr(p, "metin", "") or (p.get("metin") if isinstance(p, dict) else "")
        skor = getattr(p, "skor", None)
        if skor is None and isinstance(p, dict):
            skor = p.get("skor")
        sayfa = getattr(p, "sayfa", None)
        if sayfa is None and isinstance(p, dict):
            sayfa = p.get("sayfa")
        skor_s = f"{float(skor):.3f}" if skor is not None else "-"
        sayfa_s = f", sayfa {sayfa}" if sayfa not in (None, "") else ""
        snippet = metin if len(str(metin)) <= 700 else str(metin)[:700] + "..."
        lines.append(f"### {i}. {kanun} — {madde}")
        lines.append(f"**{baslik}**  (skor: {skor_s}{sayfa_s})")
        lines.append("")
        lines.append(snippet)
        lines.append("")
    return "\n".join(lines)



AGENT_TRACE: list = []
EVRAK_STATE: dict = {}
mcp = None


def _empty_evrak() -> dict:
    return {
        "ocr_text": "",
        "analysis": {},
        "rag_queries": [],
        "provisions": [],
        "unit_suggestion": "",
        "clerk": {},
        "draft": "",
        "flags": {"eksik_bilgi": False, "rag_bos": False},
        "trace": [],
    }


def _has_missing(analysis: dict) -> bool:
    miss = (analysis or {}).get("missing_information")
    if isinstance(miss, list):
        return any(
            str(x).strip() and str(x).strip().lower() not in {"yok", "-", "yok."}
            for x in miss
        )
    s = str(miss or "").strip()
    return bool(s) and s.lower() not in {"yok", "-"}


def _writer_lock_text(payload: dict) -> str:
    bits = []
    bits.append(
        "KILIT: draft sonuna Ek / Ekler / ek listesi / Basvuru Dilekcesi (1 sayfa) YAZMA. "
        "PERSON basvuran vatandas; sen mudurluksun, o isimle yazma/imza. "
        "Atif kapanisa/en sona yigma. requested_action yapilmis is DEGIL."
    )
    if (payload or {}).get("process_status") == "EKSIK_BILGI_BEKLENIYOR":
        bits.append(
            "KILIT: process_status=EKSIK_BILGI_BEKLENIYOR. "
            "response_type zorunlu EKSIK_BILGI_BELGE_TAMAMLAMA_YAZISI."
        )
    sel = (payload or {}).get("selected_legislation") or []
    if not sel:
        bits.append(
            "KILIT: selected_legislation BOS. Kanun adi ve madde numarasi UYDURMA. "
            "Sadece genel 'ilgili mevzuat hukumleri' de."
        )
    else:
        cites = []
        for x in sel:
            if not isinstance(x, dict):
                continue
            kanun = (x.get("kanun") or "").strip()
            madde = (x.get("madde") or "").strip()
            baslik = (x.get("baslik") or "").strip()
            if madde and "madde" not in madde.lower() and any(ch.isdigit() for ch in madde):
                madde = f"Madde {madde}"
            c = f"{kanun} {madde}".strip()
            if baslik and baslik not in {"-", "—"} and baslik.lower() not in c.lower():
                c = f"{c} ({baslik})"
            if c:
                cites.append(c)
        if cites:
            bits.append(
                "KILIT: atifi GOVDEDE yaz (kapanis/en sona yigma; "
                "(kapanis/imza satiri degil; yalnizca 'ilgili mevzuat' YASAK; "
                "listede olmayan kanun YAZMA): " + "; ".join(cites)
            )
    return "\n".join(bits)


def format_agent_board() -> str:
    st = EVRAK_STATE or {}
    flags = st.get("flags") or {}
    a = st.get("analysis") if isinstance(st.get("analysis"), dict) else {}
    topic = a.get("primary_topic") or "—"
    n_ocr = len(st.get("ocr_text") or "")
    n_q = len(st.get("rag_queries") or [])
    n_p = len(st.get("provisions") or [])
    clerk = st.get("clerk") or {}
    draft = st.get("draft") or ""
    done = {x.get("ajan") for x in (st.get("trace") or []) if isinstance(x, dict)}

    def row(n, name, key, text):
        mark = "bitti" if key in done else "bekliyor"
        return f"{n}. **{name}** — {mark}: {text}"

    oku = f"OCR {n_ocr} karakter" if n_ocr else "görsel yok"
    eks = "eksik bilgi var" if flags.get("eksik_bilgi") else "eksik yok"
    ana = f"konu `{topic}` · {eks}" if "Analiz" in done else "Qwen1 bekliyor"
    if "Mevzuat" in done:
        rag = f"{n_q} sorgu, {n_p} madde"
        if flags.get("rag_bos"):
            rag += " · RAG boş (yazıda madde uydurma yok)"
    else:
        rag = "arama yok"
    mem = clerk.get("note") or "HITL — madde işaretle, süreç seç, sonra Yazıcı"
    yaz = "taslak üretildi" if draft else "henüz yok"
    return "\n\n".join([
        "**Ajan masası** — Okuyucu → Analiz → Mevzuat → **dur (memur)** → Yazıcı. Birim tablo, ajan değil.",
        "",
        row("1", "Okuyucu", "Okuyucu", oku),
        row("2", "Analiz", "Analiz", ana),
        row("3", "Mevzuat", "Mevzuat", rag),
        f"— **Memur** — {mem}",
        row("4", "Yazıcı", "Yazıcı", yaz),
    ])


def _agent_trace(name: str, extra: str = "") -> None:
    line = name if not extra else f"{name}: {extra}"
    AGENT_TRACE.append(line)
    print("MCP:", line, flush=True)


def ocr_gorsel_oku(image_path: str) -> str:
    """Okuyucu ajanı. Belge görselinden OCR. Birim seçmez."""
    _agent_trace("Okuyucu / ocr_gorsel_oku")
    img = Image.open(image_path).convert("RGB")
    return ocr_image(img)


def evrak_analiz_et(ham_metin: str) -> str:
    """Analiz ajanı. Qwen1 LoRA JSON. target_unit / birim ÜRETMEZ."""
    _agent_trace("Analiz / evrak_analiz_et", f"{len(ham_metin or '')} kr")
    parsed = analyze_text(ham_metin)
    return json.dumps(parsed, ensure_ascii=False)


def mevzuat_ara(sorgu_metni: str) -> str:
    """Mevzuat ajanı. RAG; sorgular || ile ayrılır."""
    _agent_trace("Mevzuat / mevzuat_ara", (sorgu_metni or "")[:90])
    queries = [q.strip() for q in (sorgu_metni or "").split("||") if q.strip()]
    if not queries:
        queries = ["belediye gorev yetki mevzuat"]
    provisions = searcher.search(queries, top_k=8)
    provisions = filter_boilerplate_provisions(provisions)[:RAG_TOP_K]
    out = []
    for p in provisions:
        if isinstance(p, dict):
            out.append({
                "kanun": p.get("kanun") or "",
                "madde": p.get("madde") or "",
                "baslik": p.get("baslik") or "",
                "metin": p.get("metin") or "",
                "skor": p.get("skor"),
            })
        else:
            out.append({
                "kanun": getattr(p, "kanun", "") or "",
                "madde": getattr(p, "madde", "") or "",
                "baslik": getattr(p, "baslik", "") or "",
                "metin": getattr(p, "metin", "") or "",
                "skor": getattr(p, "skor", None),
            })
    return json.dumps(out, ensure_ascii=False)


def extract_official_letter(raw) -> str:
    text = raw if isinstance(raw, str) else json.dumps(raw, ensure_ascii=False)
    text = (text or "").strip()
    parsed = try_parse_json(text)
    if isinstance(parsed, dict):
        d = parsed.get("draft")
        if isinstance(d, str) and d.strip():
            text = d.strip()
        elif isinstance(d, dict) and isinstance(d.get("draft"), str):
            text = d["draft"].strip()
    if text.lstrip().startswith("{") and "response_type" in text:
        p2 = try_parse_json(text)
        if isinstance(p2, dict) and isinstance(p2.get("draft"), str):
            text = p2["draft"].strip()
    lines = []
    skip = False
    for ln in text.splitlines():
        low = ln.lower()
        if any(k in low for k in ("kimlik no", "t.c. kimlik", "tc kimlik")):
            skip = True
            continue
        if skip and (low.startswith("imza") or low.startswith("telefon") or low.startswith("adres") or low.startswith("ad soyad")):
            continue
        if skip and not ln.strip():
            skip = False
            continue
        if skip:
            continue
        lines.append(ln)
    return "\n".join(lines).strip()


def resmi_yazi_uret(girdi_json: str) -> str:
    """Yazıcı ajanı. process_status ve target_unit GİRDİ; birim seçilmez."""
    global EVRAK_STATE
    _agent_trace("Yazıcı / resmi_yazi_uret")
    payload = json.loads(girdi_json) if isinstance(girdi_json, str) else dict(girdi_json)
    extra = _writer_lock_text(payload)
    raw = _draft_resident(payload.get("ocr_excerpt") or "", payload, extra_system=extra)
    if not isinstance(EVRAK_STATE, dict) or not EVRAK_STATE:
        EVRAK_STATE = _empty_evrak()
    EVRAK_STATE["clerk"] = {
        "note": (
            f"süreç {payload.get('process_status')}; "
            f"işaretli madde {len(payload.get('selected_legislation') or [])}"
        ),
        "process_status": payload.get("process_status"),
    }
    EVRAK_STATE["draft"] = raw
    tr = EVRAK_STATE.setdefault("trace", [])
    if not any(isinstance(x, dict) and x.get("ajan") == "Yazıcı" for x in tr):
        tr.append({"ajan": "Yazıcı", "extra": extra[:80] if extra else "kilit yok"})
    else:
        for x in tr:
            if isinstance(x, dict) and x.get("ajan") == "Yazıcı":
                x["extra"] = extra[:80] if extra else "kilit yok"
    return raw


try:
    from fastmcp import FastMCP
    mcp = FastMCP("BelediyeEvrak")
    for _fn in (ocr_gorsel_oku, evrak_analiz_et, mevzuat_ara, resmi_yazi_uret):
        if hasattr(mcp, "add_tool"):
            mcp.add_tool(_fn)
        else:
            mcp.tool()(_fn)
    print("FastMCP: Okuyucu | Analiz | Mevzuat | Yazıcı")
    print("birim_oneri YOK — topic->mudurluk esleme tablosu, ajan degil.")
except Exception as e:
    print("FastMCP kayit atlandi (fonksiyonlar yine tool gibi cagrilir):", type(e).__name__, e)


def run_intake(image: Image.Image, status_cb=None):
    """Okuyucu -> Analiz -> Mevzuat. Yazı yok. Birim tool DEĞİL. Sonra memur."""
    global AGENT_TRACE, EVRAK_STATE
    AGENT_TRACE = []
    EVRAK_STATE = _empty_evrak()
    t0 = time.time()

    def st(msg):
        print(f"[{time.time()-t0:6.1f}s] {msg}", flush=True)
        if status_cb:
            status_cb(msg)

    tmp = Path(tempfile.mkstemp(prefix="ocr_img_", suffix=".png")[1])
    image.save(tmp, format="PNG")
    try:
        st("Okuyucu — ocr_gorsel_oku")
        ocr_text = ocr_gorsel_oku(str(tmp))
        EVRAK_STATE["ocr_text"] = ocr_text
        EVRAK_STATE["trace"].append({"ajan": "Okuyucu", "extra": f"{len(ocr_text or '')} kr"})

        st("Analiz — evrak_analiz_et")
        analysis = try_parse_json(evrak_analiz_et(ocr_text)) or {"_raw": True}
        EVRAK_STATE["analysis"] = analysis if isinstance(analysis, dict) else {}
        EVRAK_STATE["flags"]["eksik_bilgi"] = _has_missing(EVRAK_STATE["analysis"])
        EVRAK_STATE["unit_suggestion"] = lookup_unit(EVRAK_STATE["analysis"])
        EVRAK_STATE["trace"].append({"ajan": "Analiz", "extra": str(EVRAK_STATE["analysis"].get("primary_topic") or "")})

        st("Mevzuat — mevzuat_ara")
        queries = build_rag_queries(analysis if isinstance(analysis, dict) else {})
        provisions = json.loads(mevzuat_ara(" || ".join(queries)))
        EVRAK_STATE["rag_queries"] = queries
        EVRAK_STATE["provisions"] = provisions
        EVRAK_STATE["flags"]["rag_bos"] = not bool(provisions)
        EVRAK_STATE["clerk"] = {"note": "dur — memur (HITL): madde işaretle + süreç"}
        EVRAK_STATE["trace"].append({"ajan": "Mevzuat", "extra": f"{len(queries)} sorgu / {len(provisions)} madde"})
    finally:
        try:
            tmp.unlink()
        except Exception:
            pass
    st("ajan: " + " -> ".join(AGENT_TRACE) + f" | dur (memur) | {time.time()-t0:.0f}s")
    return ocr_text, analysis, queries, provisions


load_models_once()
print("Pipeline hazır. MODELS_READY=", globals().get("MODELS_READY"), "USE_SEQUENTIAL=", globals().get("USE_SEQUENTIAL"))


## 7) İsteğe bağlı hızlı test (görsel yolu)

Gradio'ya geçmeden OCR+analiz+RAG. Yazı taslağı bu hücrede üretilmez.


In [ ]:
TEST_IMAGE_PATH = None  # ornek: "/content/test_belge.jpg"

if TEST_IMAGE_PATH:
    img = Image.open(TEST_IMAGE_PATH)
    ocr_text, analysis, queries, provisions = run_intake(img)
    print("--- OCR ---")
    print(ocr_text[:1200])
    print("\n--- ANALIZ ---")
    print(json.dumps(analysis, ensure_ascii=False, indent=2)[:2000])
    print("\n--- SORGULAR ---", queries)
    print("\n--- ONERILEN BIRIM ---", lookup_unit(analysis))
    print("\n--- RAG ---")
    print(provisions_to_markdown(queries, provisions)[:1500])
    print("\n--- AJAN ---")
    print(format_agent_board())
else:
    print("TEST_IMAGE_PATH bos, Gradio hucresine gec.")


## 8) Gradio — tek ekran masa (share=True)

Üstte ajan masası; sol belge, orta memur+RAG, sağ taslak. Memurda durur.


In [ ]:
import json
import re
import traceback

import gradio as gr

# Sadece iskelet. Metin rengi HTML'de inline; global span/p EZME (beyaz-ustune-beyaz yapar).
WHITE_CSS = """
.gradio-container { background: #e8eef3 !important; max-width: none !important; width: 100% !important; }
footer { display: none !important; }
.masa-aside { background: #ffffff !important; border: 1px solid #c5d0d8 !important; border-radius: 12px !important; }
.masa-col { background: #ffffff !important; border: 1px solid #c5d0d8 !important; border-radius: 12px !important; }
button.primary, .gr-button-primary {
  background: #2b80b9 !important; color: #ffffff !important; border: 0 !important;
  font-weight: 700 !important; border-radius: 10px !important;
}
textarea, input[type="text"] {
  background: #ffffff !important; color: #1a1a1a !important;
  border: 1px solid #8a97a3 !important;
}
#tick-madde label, #tick-surec label,
#tick-madde .wrap label, #tick-surec .wrap label {
  display: flex !important; align-items: center !important; gap: 12px !important;
  width: 100% !important; min-height: 48px !important; margin: 0 0 8px !important;
  padding: 12px 14px !important; border-radius: 10px !important;
  background: #ffffff !important; color: #111111 !important;
  border: 1px solid #6b7780 !important; cursor: pointer !important;
  font-size: 14px !important; font-weight: 600 !important; line-height: 1.35 !important;
}
#tick-madde label span, #tick-surec label span,
#tick-madde label p, #tick-surec label p {
  color: #111111 !important; opacity: 1 !important;
}
#tick-madde label:has(input:checked), #tick-surec label:has(input:checked) {
  background: #2b80b9 !important; border-color: #2b80b9 !important;
}
#tick-madde label:has(input:checked) span, #tick-surec label:has(input:checked) span,
#tick-madde label:has(input:checked), #tick-surec label:has(input:checked) {
  color: #ffffff !important;
}
#tick-madde input[type="checkbox"], #tick-surec input[type="radio"] {
  width: 22px !important; height: 22px !important; min-width: 22px !important;
  accent-color: #2b80b9 !important; cursor: pointer !important;
  flex-shrink: 0 !important;
}
#tick-madde button, #tick-surec button {
  background: #ffffff !important; color: #111111 !important;
  min-height: 48px !important; border: 1px solid #6b7780 !important;
  border-radius: 10px !important; margin-bottom: 8px !important;
}
.ink, .ink * { color: #111111 !important; }
.ink { background: #ffffff !important; }
#ocr-panel, #ocr-panel .ink, #ocr-panel .html-container {
  max-height: none !important; overflow: visible !important;
}
.ocr-full { white-space: pre-wrap !important; color: #111111 !important;
  max-height: none !important; overflow: visible !important; word-break: break-word; }
.rag-score { white-space: nowrap !important; overflow: visible !important;
  min-width: 14em; display: inline-block; }
.rag-preview { white-space: pre-wrap; max-height: 11em; overflow: auto; }
.rag-full { margin-top: 8px; }
.rag-full > summary { color: #2b80b9 !important; font-weight: 700 !important;
  cursor: pointer; list-style: none; }
.rag-full > summary::-webkit-details-marker { display: none; }
.rag-sum-close { display: none; }
.rag-full[open] > summary .rag-sum-open { display: none; }
.rag-full[open] > summary .rag-sum-close { display: inline; }
.rag-card:has(details[open]) .rag-preview { display: none; }
.rag-full-body { white-space: pre-wrap; margin-top: 8px; }
"""

JS_LIGHT = """
() => {
  document.documentElement.classList.remove('dark');
  document.body.classList.remove('dark');
}
"""


def _h(title, inner):
    return (
        f'<div class="ink" style="background:#ffffff;border:1px solid #c5d0d8;border-radius:12px;'
        f'padding:12px 14px;margin:0 0 10px;color:#111111;font-family:Inter,Segoe UI,sans-serif">'
        f'<div style="font-size:11px;font-weight:700;color:#2b80b9;margin-bottom:8px">{title}</div>'
        f'<div class="ink" style="color:#111111;font-size:13px;line-height:1.45">{inner}</div></div>'
    )


def _esc(s) -> str:
    return str(s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def agent_board_html() -> str:
    st = EVRAK_STATE if isinstance(globals().get("EVRAK_STATE"), dict) else {}
    flags = st.get("flags") or {}
    a = st.get("analysis") if isinstance(st.get("analysis"), dict) else {}
    topic = a.get("primary_topic") or "—"
    n_ocr = len(st.get("ocr_text") or "")
    n_q = len(st.get("rag_queries") or [])
    n_p = len(st.get("provisions") or [])
    clerk = st.get("clerk") or {}
    draft = st.get("draft") or ""
    done = {x.get("ajan") for x in (st.get("trace") or []) if isinstance(x, dict)}

    def card(title, body):
        return (
            f'<div class="ink" style="flex:1;min-width:150px;background:#ffffff;border:1px solid #8a97a3;'
            f'border-radius:10px;padding:10px 12px;color:#111111">'
            f'<div style="color:#2b80b9;font-weight:800;font-size:12px">{_esc(title)}</div>'
            f'<div class="ink" style="color:#111111;font-size:13px;margin-top:6px;line-height:1.4">{body}</div></div>'
        )

    oku = f"bitti · OCR {n_ocr} kr" if "Okuyucu" in done else "bekliyor"
    eks = "eksik var" if flags.get("eksik_bilgi") else "eksik yok"
    ana = f"bitti · {_esc(topic)} · {eks}" if "Analiz" in done else "bekliyor"
    mev = f"bitti · {n_q} sorgu, {n_p} madde" if "Mevzuat" in done else "bekliyor"
    mem = _esc(clerk.get("note") or "HITL: madde işaretle + süreç")
    yaz = "bitti · taslak hazır" if draft else "bekliyor"
    row = (
        '<div style="display:flex;gap:8px;flex-wrap:nowrap">'
        + card("1. Okuyucu", oku)
        + card("2. Analiz", ana)
        + card("3. Mevzuat", mev)
        + card("Memur", mem)
        + card("4. Yazıcı", yaz)
        + "</div>"
    )
    return _h("Ajan masası", row)


EMPTY_BOARD = agent_board_html()
EMPTY_RAG = _h("Mevzuat", "Henüz yok. Belgeyi işle.")
EMPTY_LETTER = (
    '<div style="background:#fffdf8;border:1px solid #c4b89a;border-radius:4px;padding:20px;'
    'color:#1a1a1a;font-family:Times New Roman,Times,serif;font-size:15px;line-height:1.55;'
    'min-height:340px">Resmî yazı burada. JSON buraya yazılmaz.</div>'
)


def _prov_to_dict(p) -> dict:
    if isinstance(p, dict):
        metin = p.get("metin") or p.get("ozet") or ""
        skor = p.get("skor") if p.get("skor") is not None else p.get("reranker_skoru")
        return {"kanun": p.get("kanun") or "", "madde": p.get("madde") or "",
                "baslik": p.get("baslik") or "", "metin": metin, "skor": skor}
    return {
        "kanun": getattr(p, "kanun", "") or "",
        "madde": getattr(p, "madde", "") or "",
        "baslik": getattr(p, "baslik", "") or "",
        "metin": getattr(p, "metin", None) or getattr(p, "ozet", "") or "",
        "skor": getattr(p, "skor", None) or getattr(p, "reranker_skoru", None),
    }


def _prov_label(d: dict, i: int) -> str:
    return f"{i}. {d.get('kanun','')} — {d.get('madde','')} | {str(d.get('baslik') or '')[:70]}"


def provisions_cards_html(plist: list) -> str:
    if not plist:
        return _h("Mevzuat", "Bulunamadı veya Amaç/Kapsam/Tanım elendi.")
    lim = 280
    cards = []
    for i, d in enumerate(plist, 1):
        metin = str(d.get("metin") or "").strip() or "—"
        govde = (
            f'<div class="ink rag-preview" style="color:#111111;font-size:13px;'
            f'line-height:1.5">{_esc(metin)}</div>'
        )
        if len(metin) > lim:
            govde += (
                f'<details class="rag-full">'
                f'<summary><span class="rag-sum-open">Tam metni göster</span>'
                f'<span class="rag-sum-close">Kapat</span></summary>'
                f'<div class="ink rag-full-body" style="color:#111111;font-size:13px;'
                f'line-height:1.5">{_esc(metin)}</div></details>'
            )
        cards.append(
            f'<div class="rag-card ink" style="background:#f4f7f9;border:1px solid #8a97a3;'
            f'border-radius:10px;padding:12px 14px;margin:0 0 10px;color:#111111">'
            f'<div class="rag-score" style="color:#2b80b9;font-weight:800">'
            f'{i}. sıra</div>'
            f'<div class="ink" style="font-weight:700;color:#111111">'
            f'{_esc(d.get("kanun"))} — {_esc(d.get("madde"))}</div>'
            f'<div class="ink" style="color:#111111;margin-bottom:8px">{_esc(d.get("baslik") or "-")}</div>'
            f'{govde}</div>'
        )
    return _h("Mevzuat maddeleri", "".join(cards))


def letter_html(text: str) -> str:
    body = _esc((text or "").strip()) or "Resmî yazı boş."
    return (
        f'<div style="background:#fffdf8;border:1px solid #c4b89a;border-radius:4px;padding:22px 24px;'
        f'color:#111111;font-family:Times New Roman,Times,serif;font-size:15px;line-height:1.55;'
        f'white-space:pre-wrap;min-height:360px">{body}</div>'
    )


def _fmt_info_html(val) -> str:
    """key_information / missing_information: liste veya sözlük."""
    items = []
    if val is None or val == "":
        return "—"
    if isinstance(val, str):
        s = val.strip()
        return _esc(s) if s and s.lower() not in {"yok", "-", "yok."} else "—"
    if isinstance(val, dict):
        for k, v in val.items():
            vs = "" if v is None else str(v).strip()
            if vs and vs.lower() not in {"yok", "-", "yok."}:
                items.append((str(k), vs))
    elif isinstance(val, list):
        for it in val:
            if isinstance(it, dict):
                t = str(it.get("type") or it.get("key") or "").strip()
                v = str(it.get("value") or it.get("val") or "").strip()
                if t or (v and v.lower() not in {"yok", "-", "yok."}):
                    items.append((t or "bilgi", v or "—"))
            else:
                s = str(it).strip()
                if s and s.lower() not in {"yok", "-", "yok."}:
                    items.append(("", s))
    else:
        return _esc(str(val))
    if not items:
        return "—"
    bits = []
    for k, v in items:
        if k:
            bits.append(
                f'<div class="ink" style="margin:2px 0 0 8px;color:#111111">'
                f'• <b>{_esc(k)}</b>: {_esc(v)}</div>'
            )
        else:
            bits.append(
                f'<div class="ink" style="margin:2px 0 0 8px;color:#111111">• {_esc(v)}</div>'
            )
    return "".join(bits)


def analysis_html(analysis: dict) -> str:
    if not isinstance(analysis, dict) or analysis.get("_parse_error"):
        return _h("Analiz", "Yok.")
    simple = [
        ("Belge türü", analysis.get("document_type")),
        ("Gönderen", analysis.get("sender_type")),
        ("Konu", analysis.get("primary_topic")),
        ("Talep (requested_action)", analysis.get("requested_action")),
        ("Özet", analysis.get("summary")),
    ]
    bits = "".join(
        f'<div class="ink" style="margin:0 0 6px;color:#111111"><b>{_esc(k)}:</b> {_esc(v or "—")}</div>'
        for k, v in simple
    )
    bits += (
        f'<div class="ink" style="margin:8px 0 4px;color:#111111"><b>Ana bilgiler (key_information):</b></div>'
        + _fmt_info_html(analysis.get("key_information"))
        + f'<div class="ink" style="margin:8px 0 4px;color:#111111"><b>Eksik bilgi (missing_information):</b></div>'
        + _fmt_info_html(analysis.get("missing_information"))
    )
    return _h("Analiz", bits)


def response_type_html(rt: str) -> str:
    t = (rt or "").strip() or "—"
    return _h("Yazı türü (response_type)", f'<div class="ink" style="font-weight:700;color:#111111">{_esc(t)}</div>')


EMPTY_RESP = response_type_html("")


def ocr_html(text: str) -> str:
    return (
        '<div id="ocr-panel" class="ink" style="background:#ffffff;border:1px solid #c5d0d8;'
        'border-radius:12px;padding:12px 14px;margin:0 0 10px;color:#111111">'
        '<div style="font-size:11px;font-weight:700;color:#2b80b9;margin-bottom:8px">OCR</div>'
        f'<div class="ocr-full">{_esc(text or "—")}</div></div>'
    )


def status_html(text: str) -> str:
    return _h("Durum", _esc(text or "—"))


def lookup_unit(analysis: dict) -> str:
    topic = str((analysis or {}).get("primary_topic") or "").strip()
    return TOPIC_TO_UNIT.get(topic, "Yazı İşleri Müdürlüğü")


def _cite_label(d: dict) -> str:
    """Kanun adı + Madde no + madde başlığı (hardcode atıfın havada kalmaması için)."""
    kanun = (d.get("kanun") or "").strip()
    madde = (d.get("madde") or "").strip()
    baslik = (d.get("baslik") or "").strip()
    if baslik in {"-", "—", "–", "yok", "Yok"}:
        baslik = ""
    if madde and not re.search(r"(?i)\bmadde\b", madde) and re.search(r"\d+", madde):
        madde = f"Madde {madde}"
    bits = [x for x in (kanun, madde) if x]
    if baslik:
        blob = " ".join(bits).lower()
        if baslik.lower() not in blob and len(baslik) <= 90:
            bits.append(f"({baslik})")
    return " ".join(bits).strip()


def _strip_dative_unit(s: str) -> str:
    t = (s or "").strip()
    for a, b in (
        ("Müdürlüğüne", "Müdürlüğü"),
        ("MÜDÜRLÜĞÜNE", "MÜDÜRLÜĞÜ"),
        ("Başkanlığına", "Başkanlığı"),
        ("BAŞKANLIĞINA", "BAŞKANLIĞI"),
        ("müdürlüğüne", "müdürlüğü"),
        ("başkanlığına", "başkanlığı"),
    ):
        if t.endswith(a):
            return t[: -len(a)] + b
    return t


def _citizen_tokens(analysis: dict) -> list:
    """Başvuran kişi/adres kelimeleri. key_information hem liste hem sözlük olabilir."""
    toks = []
    ki = (analysis or {}).get("key_information") if isinstance(analysis, dict) else None
    pairs = []
    if isinstance(ki, list):
        for it in ki:
            if isinstance(it, dict):
                pairs.append((str(it.get("type") or ""), str(it.get("value") or "")))
    elif isinstance(ki, dict):
        for k, v in ki.items():
            pairs.append((str(k), str(v or "")))
    want = ("person", "ad", "soyad", "isim", "basvuran", "kisi",
            "location", "address", "adres", "mahalle", "konum")
    for t, v in pairs:
        tl = t.lower()
        vv = v.strip()
        if not vv or vv.lower() in {"yok", "-", "yok."}:
            continue
        if any(x in tl for x in want):
            toks.append(vv)
            for w in re.split(r"[,\s/]+", vv):
                if len(w) > 2:
                    toks.append(w)
    return toks


_EK_HEAD = re.compile(
    r"(?i)^\s*ek(?:ler| listesi)?\s*:?\s*(.*)$"
)


def _is_ek_item_line(s: str) -> bool:
    t = (s or "").strip()
    if not t:
        return False
    if t.startswith(("-", "•", "*", "–")) and re.search(
        r"(?i)(dilekçe|sayfa|evrak|fotokopi|başvuru)", t
    ):
        return True
    m = _EK_HEAD.match(t)
    if not m:
        return False
    rest = (m.group(1) or "").strip()
    return (not rest) or bool(re.search(r"(?i)(dilekçe|sayfa|evrak|fotokopi|başvuru)", rest))


def _strip_trailing_ek(text: str) -> str:
    """Modelin yazı sonuna yapıştırdığı sahte Ek / dilekçe listesini kes."""
    lines = (text or "").rstrip().splitlines()
    while lines and (not lines[-1].strip() or _is_ek_item_line(lines[-1])):
        lines.pop()
    close_i = None
    for i, ln in enumerate(lines):
        low = ln.strip().lower()
        if low.startswith(("saygılarımla", "saygılar", "saygilar", "bilgilerinize", "gereğini", "geregini")):
            close_i = i
        elif "rica ederim" in low or "arz ederim" in low:
            close_i = i
    if close_i is not None:
        head, tail = lines[: close_i + 1], lines[close_i + 1 :]
        out_tail, skipping = [], False
        for ln in tail:
            if _is_ek_item_line(ln) or (skipping and not ln.strip()):
                skipping = True
                continue
            if skipping and ln.strip().startswith(("-", "•", "*", "–")):
                continue
            skipping = False
            out_tail.append(ln)
        lines = head + out_tail
    return "\n".join(lines).rstrip()


def polish_official_letter(letter: str, payload: dict, analysis: dict, sender_unit: str) -> str:
    """Belediye → vatandaş: yanlış antet/hitap ve vatandaş imzasını kes.

    ensure_legislation_in_draft'tan ÖNCE çağrılmalı; yoksa atıf
    kapanış/imza satırlarına yapışır.
    """
    text = (letter or "").strip()
    sender = _strip_dative_unit(sender_unit or (payload or {}).get("target_unit") or "")
    dest = _strip_dative_unit((payload or {}).get("target_unit") or "")
    lines = text.splitlines()
    i = 0
    while i < min(len(lines), 8):
        ln = lines[i].strip()
        up = ln.upper()
        if (not ln) or up in {"T.C.", "TC"} or up.endswith("BAŞKANLIĞINA") or up.endswith("MÜDÜRLÜĞÜNE"):
            i += 1
            continue
        break
    body = "\n".join(lines[i:]).lstrip()
    head = "\n".join(lines[:i])
    if re.search(r"(BAŞKANLIĞINA|MÜDÜRLÜĞÜNE)", head, re.I) or not re.match(r"(?i)^\s*T\.C\.", text):
        text = f"T.C.\n{sender}\n\n{body}".strip() if sender else f"T.C.\n\n{body}".strip()

    # Antette birimin parantezli tekrarını at: "(Veteriner İşleri Müdürlüğü)"
    slow = sender.lower()
    cleaned, prev = [], None
    for ln in text.splitlines():
        s = ln.strip()
        bare = s.strip("()").strip().lower()
        if s.startswith("(") and s.endswith(")") and bare == slow:
            continue
        if prev is not None and s and s == prev:
            continue
        cleaned.append(ln)
        if s:
            prev = s
    text = "\n".join(cleaned)

    # Vatandaş imza bloğunu sondan kes (isim + adres + telefon).
    toks = [t.lower() for t in _citizen_tokens(analysis if isinstance(analysis, dict) else {}) if len(t) > 2]
    phone = re.compile(r"(?<!\d)(0\s*)?5\d[\d\s]{8,}")
    addr = re.compile(r"(?i)(mahallesi|caddesi|sokak|sok\.|no:|telefon|/[A-ZÇĞİÖŞÜ])")
    out_lines = text.rstrip().splitlines()
    while out_lines:
        ln = out_lines[-1].strip()
        if not ln:
            out_lines.pop()
            continue
        low = ln.lower()
        # Kapanış selamı geldiyse imza bloğu bitmiştir, dur.
        if low.startswith(("saygılar", "saygilar", "bilgilerinize", "gereğini", "geregini")):
            break
        if toks and any(tok in low for tok in toks):
            out_lines.pop()
            continue
        if phone.search(ln) or addr.search(ln):
            out_lines.pop()
            continue
        break
    text = "\n".join(out_lines).rstrip()
    text = _strip_trailing_ek(text)

    if (payload or {}).get("process_status") == "YONLENDIRILDI" and dest:
        if dest.lower() not in text.lower():
            text = text.rstrip() + f"\n\nBaşvurunuz {dest} birimine yönlendirilmiştir.\n"
    return text


def _is_closing_line(ln: str) -> bool:
    low = (ln or "").strip().lower()
    if not low:
        return False
    if low.startswith(("saygılarımla", "saygılar", "saygilar", "bilgilerinize", "gereğini", "geregini")):
        return True
    return "rica ederim" in low or "arz ederim" in low


def _is_headerish_line(ln: str) -> bool:
    s = (ln or "").strip()
    if not s:
        return True
    low = s.lower()
    if low in {"t.c.", "tc"}:
        return True
    if low.startswith(("sayın", "sayin", "ilgi", "konu", "t.c")):
        return True
    if len(s) < 80 and re.search(r"(Müdürlüğü|Müdürlüğü\.|Başkanlığı)$", s):
        return True
    return False


def _strip_trail_islemler(para: str) -> str:
    return re.sub(
        r"\s*İşlemler\s+.+?uyarınca(?:\s+yürütülmüştür)?\.?\s*$",
        "",
        (para or "").rstrip(),
        flags=re.I | re.DOTALL,
    ).rstrip()


def _insert_cite_into_body(draft: str, sentence: str) -> str:
    """Model atıf yazmadıysa kapanış selamından hemen önce ayrı paragraf ekle.

    En tepeye (T.C./Sayı/Konu/İlgi üstüne) veya imzaya asla girmez.
    """
    text = (draft or "").rstrip()
    clause = (sentence or "").strip().rstrip(".")
    if not clause:
        return draft or ""
    cite_para = f"İşlemler {clause} yürütülmüştür."
    if not text:
        return cite_para + "\n"
    lines = text.splitlines()

    # Model zaten gövdede atıf yaptıysa dokunma.
    for ln in lines:
        lk = ln.lower()
        if ("uyarınca" in lk or "gereğince" in lk) and "madde" in lk and not _is_closing_line(ln):
            return "\n".join(lines).rstrip() + "\n"

    close_i = next((i for i, ln in enumerate(lines) if _is_closing_line(ln)), None)

    # Kapanıştan önceki son gövde satırındaki yarım "İşlemler ... uyarınca" kalıntısını temizle.
    body_end = close_i if close_i is not None else len(lines)
    j = body_end - 1
    while j >= 0 and not lines[j].strip():
        j -= 1
    if j >= 0:
        cleaned = _strip_trail_islemler(lines[j])
        lines[j] = cleaned

    if close_i is None:
        return "\n".join(lines).rstrip() + "\n\n" + cite_para + "\n"

    ins = close_i
    while ins - 1 >= 0 and not lines[ins - 1].strip():
        ins -= 1
    lines[ins:ins] = ["", cite_para, ""]
    return "\n".join(lines).rstrip() + "\n"


def ensure_legislation_in_draft(draft: str, chosen: list) -> str:
    if not chosen:
        return draft or ""
    cites = ", ".join(_cite_label(d) for d in chosen[:2] if _cite_label(d))
    if not cites:
        return draft or ""
    return _insert_cite_into_body(draft, f"{cites} uyarınca")


def intake_handler(image):
    empty_cb = gr.update(choices=[], value=[])
    empty = (
        EMPTY_BOARD, ocr_html("Önce görsel yükle."), analysis_html({}), "{}",
        "", "Yazı İşleri Müdürlüğü", EMPTY_RAG, empty_cb,
        "INCELEMEDE", gr.update(value="", visible=False), "", "", None, None, [],
        EMPTY_RESP,
    )
    if image is None:
        return empty
    def st(msg):
        print(msg, flush=True)
    try:
        ocr_text, analysis, queries, provisions = run_intake(image, status_cb=st)
    except Exception:
        err = traceback.format_exc()
        print(err, flush=True)
        return (
            EMPTY_BOARD, ocr_html(err), analysis_html({}), "{}",
            "", "Yazı İşleri Müdürlüğü", _h("Hata", _esc(err)), empty_cb,
            "INCELEMEDE", gr.update(value="", visible=False), "", "", None, None, [],
            EMPTY_RESP,
        )
    plist = [_prov_to_dict(p) for p in (provisions or [])]
    labels = [_prov_label(d, i) for i, d in enumerate(plist, 1)]
    unit = lookup_unit(analysis)
    eksik = _has_missing(analysis if isinstance(analysis, dict) else {})
    # Eksik bilgi -> surec EKSIK_BILGI_BEKLENIYOR; mevzuat secimi kilitli ve bos (Qwen2'ye gitmez).
    default_sel = [] if eksik else (labels[:2] if labels else [])
    default_proc = "EKSIK_BILGI_BEKLENIYOR" if eksik else "INCELEMEDE"
    return (
        agent_board_html(),
        ocr_html(ocr_text),
        analysis_html(analysis if isinstance(analysis, dict) else {}),
        json.dumps(analysis, ensure_ascii=False, indent=2),
        " | ".join(queries or []) or "(sorgu yok)",
        unit,
        provisions_cards_html(plist),
        gr.update(choices=labels, value=default_sel, interactive=not eksik),
        default_proc, gr.update(value="", visible=False), "", "",
        ocr_text, analysis, plist,
        EMPTY_RESP,
    )


def _on_proc_change(st):
    """Yönlendirme kutusunu göster/gizle + eksik bilgi ise mevzuat seçimini kilitle."""
    yon = gr.update(visible=(st == "YONLENDIRILDI"))
    if st == "EKSIK_BILGI_BEKLENIYOR":
        cb = gr.update(interactive=False, value=[])
    else:
        cb = gr.update(interactive=True)
    return yon, cb


def draft_handler(selected, process_status, performed_raw, result_info, unit, yon_unit, ocr_s, analysis_s, plist):
    if not analysis_s:
        return "{}", EMPTY_LETTER, "{}", EMPTY_BOARD, EMPTY_RESP
    actions = [x.strip() for x in (performed_raw or "").split("\n") if x.strip()]
    selected = selected or []
    status = process_status or "INCELEMEDE"
    chosen = []
    # Eksik bilgi yazisinda mevzuat Qwen2'ye gitmez.
    if status != "EKSIK_BILGI_BEKLENIYOR" and plist:
        for i, d in enumerate(plist, 1):
            if _prov_label(d, i) in selected:
                chosen.append(d)
    dest = (yon_unit or "").strip()
    if status == "YONLENDIRILDI" and dest:
        target = dest
    else:
        target = unit or lookup_unit(analysis_s)
    payload_in = {
        "document_type": analysis_s.get("document_type", ""),
        "sender_type": analysis_s.get("sender_type", ""),
        "primary_topic": analysis_s.get("primary_topic", ""),
        "requested_action": analysis_s.get("requested_action", ""),
        "key_information": analysis_s.get("key_information", {}),
        "missing_information": analysis_s.get("missing_information", []),
        "summary": analysis_s.get("summary", ""),
        "process_status": status,
        "performed_actions": actions,
        "result_information": (result_info or "").strip(),
        "target_unit": target,
        "selected_legislation": [
            {k: x.get(k) for k in ("kanun", "madde", "baslik", "metin")}
            for x in chosen
        ],
    }
    try:
        raw = resmi_yazi_uret(json.dumps(payload_in, ensure_ascii=False))
    except Exception:
        err = traceback.format_exc()
        print(err, flush=True)
        return json.dumps(payload_in, ensure_ascii=False, indent=2), letter_html("HATA:\n" + err), json.dumps(payload_in, ensure_ascii=False, indent=2), EMPTY_BOARD, EMPTY_RESP
    parsed = try_parse_json(raw) or {}
    letter = extract_official_letter(raw)
    if payload_in["process_status"] == "EKSIK_BILGI_BEKLENIYOR" and isinstance(parsed, dict):
        parsed["response_type"] = "EKSIK_BILGI_BELGE_TAMAMLAMA_YAZISI"
    letter = polish_official_letter(letter, payload_in, analysis_s, sender_unit=unit or lookup_unit(analysis_s))
    letter = ensure_legislation_in_draft(letter, chosen)
    out_show = {
        "response_type": parsed.get("response_type") if isinstance(parsed, dict) else "",
        "target_unit": payload_in["target_unit"],
        "process_information": parsed.get("process_information") if isinstance(parsed, dict) else "",
        "draft": letter,
    }
    return (
        json.dumps(out_show, ensure_ascii=False, indent=2),
        letter_html(letter),
        json.dumps(payload_in, ensure_ascii=False, indent=2),
        agent_board_html(),
        response_type_html(out_show.get("response_type") or ""),
    )


def rag_query_handler(query):
    q = (query or "").strip()
    if not q:
        return _h("Mevzuat", "Sorgu yaz.")
    try:
        provisions = searcher.search([q], top_k=RAG_TOP_K)
        provisions = filter_boilerplate_provisions(provisions)[:RAG_TOP_K]
    except Exception:
        return _h("Hata", _esc(traceback.format_exc()))
    return provisions_cards_html([_prov_to_dict(p) for p in (provisions or [])])


# Radio değeri enum (mantık için); ekranda düzgün Türkçe etiket.
PROCESS_STATUS_LABELS = {
    "INCELEMEDE": "İnceleniyor",
    "TAMAMLANDI": "Tamamlandı",
    "EKSIK_BILGI_BEKLENIYOR": "Eksik bilgi bekleniyor",
    "REDDEDILDI": "Reddedildi",
    "YONLENDIRILDI": "Yönlendirildi",
}
PROC_STATUS_CHOICES = [
    (PROCESS_STATUS_LABELS.get(v, v), v) for v in PROCESS_STATUS_CHOICES
]


_theme = gr.themes.Base(primary_hue="blue", secondary_hue="teal", neutral_hue="slate")
try:
    _theme = _theme.set(
        body_background_fill="#e8eef3",
        body_background_fill_dark="#e8eef3",
        body_text_color="#1a1a1a",
        body_text_color_dark="#1a1a1a",
        block_background_fill="#ffffff",
        block_background_fill_dark="#ffffff",
        block_label_text_color="#1a1a1a",
        block_label_text_color_dark="#1a1a1a",
        block_title_text_color="#1a1a1a",
        block_title_text_color_dark="#1a1a1a",
        input_background_fill="#ffffff",
        input_background_fill_dark="#ffffff",
        input_text_color="#1a1a1a",
        input_text_color_dark="#1a1a1a",
        button_primary_text_color="#ffffff",
        button_primary_background_fill="#2b80b9",
    )
except Exception:
    pass

_kw = dict(title="KADİM Evrak Masası", theme=_theme, css=WHITE_CSS)
try:
    demo_ctx = gr.Blocks(js=JS_LIGHT, head='<meta name="color-scheme" content="light">', fill_width=True, **_kw)
except TypeError:
    try:
        demo_ctx = gr.Blocks(js=JS_LIGHT, head='<meta name="color-scheme" content="light">', **_kw)
    except TypeError:
        demo_ctx = gr.Blocks(**_kw)

with demo_ctx as demo:
    agent_md = gr.HTML(EMPTY_BOARD)
    st_ocr = gr.State("")
    st_analysis = gr.State(None)
    st_plist = gr.State([])
    with gr.Row():
        with gr.Column(elem_classes=["masa-col"], scale=4, min_width=320):
            gr.HTML('<div style="color:#1a1a1a;font-weight:800;font-size:16px">Belge</div>')
            img_in = gr.Image(type="pil", label="Görsel", height=220)
            btn_in = gr.Button("Belgeyi işle (OCR + analiz + RAG)", variant="primary")
            ocr_out = gr.HTML(ocr_html(""), elem_id="ocr-panel")
            analysis_md = gr.HTML(analysis_html({}))
            with gr.Accordion("Ham JSON (analiz)", open=False):
                analysis_out = gr.Code(language="json", lines=6)
        with gr.Column(elem_classes=["masa-col"], scale=5, min_width=360):
            gr.HTML('<div style="color:#1a1a1a;font-weight:800;font-size:16px">Memur + mevzuat</div>')
            unit_out = gr.Textbox(label="Hedef birim", lines=1)
            rag_q_out = gr.Textbox(label="RAG sorguları", lines=2)
            rag_md = gr.HTML(EMPTY_RAG)
            rag_cb = gr.CheckboxGroup(choices=[], label="Yazıya gidecek maddeler", elem_id="tick-madde")
            proc = gr.Radio(PROC_STATUS_CHOICES, value="INCELEMEDE", label="Süreç", elem_id="tick-surec")
            yon_unit = gr.Textbox(
                label="Yönlendirilecek birim",
                lines=1,
                visible=False,
                placeholder="Örn. Park ve Bahçeler Müdürlüğü",
            )
            actions_in = gr.Textbox(label="Aksiyonlar (satır satır)", lines=3)
            result_in = gr.Textbox(label="Süreç sonucu", lines=2)
        with gr.Column(elem_classes=["masa-col"], scale=5, min_width=360):
            gr.HTML('<div style="color:#1a1a1a;font-weight:800;font-size:16px">Taslak</div>')
            btn_draft = gr.Button("Taslak üret", variant="primary")
            resp_type_md = gr.HTML(EMPTY_RESP)
            draft_out = gr.HTML(EMPTY_LETTER)
            with gr.Accordion("Ham JSON (yazı değil)", open=False):
                writer_in_json = gr.Code(language="json", lines=6)
                writer_json = gr.Code(language="json", lines=6)
    with gr.Accordion("Serbest mevzuat", open=False):
        rag_q = gr.Textbox(label="Sorgu", lines=2)
        rag_btn = gr.Button("Ara")
        rag_direct_out = gr.HTML()
        rag_btn.click(rag_query_handler, inputs=rag_q, outputs=rag_direct_out)
    proc.change(_on_proc_change, inputs=proc, outputs=[yon_unit, rag_cb])
    btn_in.click(
        intake_handler, inputs=img_in,
        outputs=[
            agent_md, ocr_out, analysis_md, analysis_out, rag_q_out,
            unit_out, rag_md, rag_cb, proc, yon_unit, actions_in, result_in,
            st_ocr, st_analysis, st_plist, resp_type_md,
        ],
    )
    btn_draft.click(
        draft_handler,
        inputs=[rag_cb, proc, actions_in, result_in, unit_out, yon_unit, st_ocr, st_analysis, st_plist],
        outputs=[writer_json, draft_out, writer_in_json, agent_md, resp_type_md],
    )

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=False)


---

## Notlar

- Yazı: aynı Instruct vLLM + `yazi_lora` (ikinci vLLM yok).
- Dört ajan ortak `EVRAK_STATE`; ekranda ajan masası. FastMCP kayıt var, `mcp.run` yok.
- `target_unit`: Qwen seçmez; `primary_topic` → müdürlük tablosu.
- Eksik bilgi → süreç varsayılanı `EKSIK_BILGI_BEKLENIYOR`; mevzuat kutusu seçilmez/kilitlenir, Qwen2’ye madde gitmez.
- Jüri öncesi Run all ile ısıt; ilk belge hariç model yüklenmez.
